In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [12]:
from google.colab import files

archivos = files.upload()

print("Archivos cargados:")
for nombre in archivos:
    print(nombre)

Saving testdata.manual.2009.06.14.csv to testdata.manual.2009.06.14.csv
Saving training.1600000.processed.noemoticon.csv to training.1600000.processed.noemoticon.csv
Archivos cargados:
testdata.manual.2009.06.14.csv
training.1600000.processed.noemoticon.csv


In [13]:
# Archivo original
archivo = "training.1600000.processed.noemoticon.csv"


In [14]:
# Nombres de las columnas
columnas = ["sentimiento","id","fecha","consulta","usuario","texto"]


In [15]:
# Configuración de la muestra
tamano_muestra = 100000
semilla = 42

# Selección aleatoria de 50.000 tweets por sentimiento
muestras = {0: [], 4: []}

for bloque in pd.read_csv(
    archivo,
    header=None,
    names=columnas,
    encoding="latin-1",
    chunksize=50000):
    for sentimiento in muestras:
        grupo = bloque[bloque["sentimiento"] == sentimiento]

        if not grupo.empty:
            muestras[sentimiento].append(
                grupo.sample(
                    frac=0.0625,
                    random_state=semilla))


In [16]:
# Unificar los registros seleccionados
datos = pd.concat([muestra for grupo in muestras.values() for muestra in grupo],ignore_index=True)

# Seleccionar exactamente 50.000 registros por clase
df_entrenamiento = (
    datos.groupby("sentimiento", group_keys=False)
    .sample(n=tamano_muestra // 2, random_state=semilla)
    .sample(frac=1, random_state=semilla)
    .reset_index(drop=True))

# Guardar la muestra
df_entrenamiento.to_csv("tweets_muestra_100k.csv",index=False)

print("Cantidad de tweets:", len(df_entrenamiento))
print("\nDistribución de sentimientos:")
print(df_entrenamiento["sentimiento"].value_counts())

Cantidad de tweets: 100000

Distribución de sentimientos:
sentimiento
4    50000
0    50000
Name: count, dtype: int64


### Selección de una muestra de datos

Debido al tamaño del conjunto original, se decidió trabajar
con una muestra aleatoria de aproximadamente 100.000 tweets.

El objetivo es reducir el consumo de memoria y el tiempo de
procesamiento, manteniendo un volumen suficiente de registros
para realizar el análisis exploratorio y evaluar modelos.

Se utilizó una semilla aleatoria de 42 para facilitar la
reproducibilidad del procedimiento.

Posteriormente se verificará la distribución de las clases
para evaluar la representatividad de la muestra.

### 1.1. Estructura del conjunto de datos

Se analiza la estructura de la muestra seleccionada, identificando
la cantidad de registros, las variables disponibles y sus tipos de datos.

También se visualizan los primeros registros para comprender
la información contenida en cada tweet.

In [17]:
# Dimensiones de la muestra
print("Dimensiones del conjunto de datos:")
print(df_entrenamiento.shape)

# Nombres de las variables
print("\nVariables disponibles:")
print(df_entrenamiento.columns.tolist())

# Tipos de datos
print("\nInformación general:")
df_entrenamiento.info()

# Primeros registros
print("\nPrimeros cinco registros:")
display(df_entrenamiento.head())

Dimensiones del conjunto de datos:
(100000, 6)

Variables disponibles:
['sentimiento', 'id', 'fecha', 'consulta', 'usuario', 'texto']

Información general:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 6 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   sentimiento  100000 non-null  int64 
 1   id           100000 non-null  int64 
 2   fecha        100000 non-null  object
 3   consulta     100000 non-null  object
 4   usuario      100000 non-null  object
 5   texto        100000 non-null  object
dtypes: int64(2), object(4)
memory usage: 4.6+ MB

Primeros cinco registros:


,sentimiento,id,fecha,consulta,usuario,texto
0,4,1964895956,Fri May 29 14:59:19 PDT 2009,NO_QUERY,thomscottart,15th piece for Neuwerld sent and waiting appro...
1,4,1994295873,Mon Jun 01 11:43:00 PDT 2009,NO_QUERY,mohawksteveo,no twat for 2 days! must have gotten my ass ki...
2,0,2192879825,Tue Jun 16 07:41:04 PDT 2009,NO_QUERY,rbarbera,We're getting several confirmations that some ...
3,4,2178789227,Mon Jun 15 08:12:24 PDT 2009,NO_QUERY,xMileyCyrusTeam,Please follow!!
4,4,2016281867,Wed Jun 03 06:30:14 PDT 2009,NO_QUERY,thebudget,getting ready to have coffee with a good friend


In [18]:
import pandas as pd

# Recuperar la muestra de entrenamiento
df_entrenamiento = pd.read_csv(
    "tweets_muestra_100k.csv",
    encoding="utf-8"
)

# Verificar la carga
print("Dimensiones:", df_entrenamiento.shape)
print("\nColumnas:", df_entrenamiento.columns.tolist())

Dimensiones: (100000, 6)

Columnas: ['sentimiento', 'id', 'fecha', 'consulta', 'usuario', 'texto']


In [19]:
# Valores faltantes
print("Valores faltantes por variable:")
print(df_entrenamiento.isnull().sum())

# Registros duplicados
print("\nRegistros completamente duplicados:")
print(df_entrenamiento.duplicated().sum())

# Identificadores duplicados
print("\nIdentificadores duplicados:")
print(df_entrenamiento["id"].duplicated().sum())

# Textos duplicados
print("\nTextos duplicados:")
print(df_entrenamiento["texto"].duplicated().sum())

# Textos vacíos
print("\nTextos vacíos:")
print(
    df_entrenamiento["texto"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

Valores faltantes por variable:
sentimiento    0
id             0
fecha          0
consulta       0
usuario        0
texto          0
dtype: int64

Registros completamente duplicados:
0

Identificadores duplicados:
7

Textos duplicados:
318

Textos vacíos:
0


In [20]:
# Identificar todos los registros con ID repetido
identificadores_duplicados = df_entrenamiento[
    df_entrenamiento.duplicated(subset=["id"], keep=False)
].sort_values("id")

# Cantidad de registros involucrados
print("Registros con identificadores repetidos:")
print(len(identificadores_duplicados))

# Mostrar los registros
display(
    identificadores_duplicados[
        ["id", "sentimiento", "usuario", "texto"]
    ]
)

Registros con identificadores repetidos:
14


,id,sentimiento,usuario,texto
355,1685502710,0,attilafarkas1,got up. Hey everyone! Whats up? I wanna go to ...
41516,1685502710,4,attilafarkas1,got up. Hey everyone! Whats up? I wanna go to ...
37494,1834777946,0,Natalie__xo,@jolliehaybowes You looked really pretty at so...
43765,1834777946,4,Natalie__xo,@jolliehaybowes You looked really pretty at so...
36993,1881916233,4,theSuda,#jaljeera can't beat #followfriday More Peopl...
20293,1881916233,0,theSuda,#jaljeera can't beat #followfriday More Peopl...
40286,1981071206,4,smileymango,@alicebates LOLO Awwww thats so cute. I want ...
3642,1981071206,0,smileymango,@alicebates LOLO Awwww thats so cute. I want ...
56466,1984396408,4,tyefighter,@WH1SKS sorry i'm in bed already. I don't sl...
43532,1984396408,0,tyefighter,@WH1SKS sorry i'm in bed already. I don't sl...


In [22]:
# Analizar los identificadores repetidos
revision_duplicados = (
    identificadores_duplicados
    .groupby("id")
    .agg(
        cantidad_registros=("id", "size"),
        usuarios_diferentes=("usuario", "nunique"),
        textos_diferentes=("texto", "nunique"),
        sentimientos_diferentes=("sentimiento", "nunique"),
        fechas_diferentes=("fecha", "nunique")
    )
)

display(revision_duplicados)

,cantidad_registros,usuarios_diferentes,textos_diferentes,sentimientos_diferentes,fechas_diferentes
id,,,,,
1685502710,2,1,1,2,1
1834777946,2,1,1,2,1
1881916233,2,1,1,2,1
1981071206,2,1,1,2,1
1984396408,2,1,1,2,1
2052948151,2,1,1,2,1
2189659721,2,1,1,2,1


### 1.3. Análisis de usuarios y textos repetidos

Se analiza la frecuencia de publicación de los usuarios y
la presencia de textos idénticos publicados por diferentes cuentas.

El objetivo es identificar patrones de repetición y evaluar
si estos registros requieren algún tratamiento durante
el preprocesamiento.

In [21]:
# 1. Usuarios que publicaron más de un tweet
tweets_por_usuario = df_entrenamiento.groupby("usuario").size()

print("Cantidad de usuarios únicos:")
print(df_entrenamiento["usuario"].nunique())

print("\nUsuarios que publicaron más de un tweet:")
print((tweets_por_usuario > 1).sum())

print("\nLos 10 usuarios con más tweets:")
display(
    tweets_por_usuario.sort_values(ascending=False).head(10)
)

# 2. Textos publicados por diferentes usuarios
usuarios_por_texto = df_entrenamiento.groupby("texto")["usuario"].nunique()

textos_compartidos = usuarios_por_texto[
    usuarios_por_texto > 1
].sort_values(ascending=False)

print("\nTextos publicados por diferentes usuarios:")
print(len(textos_compartidos))

print("\nLos 10 textos publicados por más usuarios:")
display(textos_compartidos.head(10))

Cantidad de usuarios únicos:
82846

Usuarios que publicaron más de un tweet:
10825

Los 10 usuarios con más tweets:


,0
usuario,
lost_dog,40
webwoke,25
mcraddictal,21
_magic8ball,21
tweetpet,20
VioletsCRUK,17
nuttychris,16
SallytheShizzle,16
linnetwoods,15



Textos publicados por diferentes usuarios:
153

Los 10 textos publicados por más usuarios:


,usuario
texto,
Good morning,8
headache,8
Good morning!,7
Goodnight,7
Headache,6
I miss you,6
at work,6
Good Morning,5
tummy ache,5
